In [14]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import linear_sum_assignment
from dataretrieval import nwis
import optuna
import os
import warnings
import logging
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
logging.basicConfig(level = logging.INFO, format = "%(message)s")
log = logging.getLogger(__name__)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re


In [15]:
dir = os.path.join("..", "Data")
seed = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cameras = {
    "Boquillas": 8,
    "CharlestonMesquite": 2,
    "Contention": 6,
    "Fairbank": 1,
    "Hereford": 5,
    "Hunter": 4,
    "Moson": 3,
    "St.David": 7,
}
camera_sites = set(cameras.keys())

target = "fp"
predictors = ["camera", "prcp", "vpd", "tmax", "tmin", "dtgw", "et", "q"]

test_frac = 0.2
tune_frac = 0.2


In [16]:
def load_climate():
    path = os.path.join(dir, "Daymet2006_2025.csv")
    climate = pd.read_csv(path)
    climate.columns = climate.columns.str.strip().str.lower()
    climate["date"] = pd.to_datetime(climate["date"], format = "mixed")
    climate = climate.rename(columns = {"tvpd": "vpd"})
    col = ["site", "camera", "date", "prcp", "vpd", "tmax", "tmin"]
    return climate[col].copy()

def load_et():
    path = os.path.join(dir, "Jawad_ET", "J.csv")
    et = pd.read_csv(path)
    et.columns = et.columns.str.strip().str.lower()
    et["date"] = pd.to_datetime(et["date"], format = "mixed")
    return et[["site", "camera", "date", "et"]].copy()

def load_fp():
    path = os.path.join(dir, "USPPFlowMonitoring2006_2025.csv")
    flow = pd.read_csv(path)
    flow.columns = flow.columns.str.strip().str.lower()
    flow["date"] = pd.to_datetime(flow["date"], format = "mixed")
    flow = flow.rename(columns = {"flow code": "fc"})
    flow["fc"] = pd.to_numeric(flow["fc"], errors = "coerce")
    flow[target] = np.where(flow["fc"].isna(), np.nan, (flow["fc"] > 0).astype(int))
    return flow.groupby(["site", "date"], as_index = False)[target].max()

def load_camera_locations():
    path = os.path.join(dir, "USPPFlowMonitoring2006_2025.csv")
    flow = pd.read_csv(path)
    flow.columns = flow.columns.str.strip().str.lower()
    locs = flow.groupby("site", as_index = False)[["latitude", "longitude"]].first()
    return locs[locs["site"].isin(camera_sites)].reset_index(drop = True)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def pair_gages_to_cameras(gage_numbers):
    """Assigns each stream gage to its nearest camera with no camera used twice (Hungarian algorithm)."""
    site_info = nwis.get_record(sites = gage_numbers, service = "site").set_index("site_no")
    cam_locs = load_camera_locations()

    dist = np.array([
        [
            haversine_km(site_info.loc[g, "dec_lat_va"], site_info.loc[g, "dec_long_va"], row["latitude"], row["longitude"])
            for _, row in cam_locs.iterrows()
        ]
        for g in gage_numbers
    ])
    gage_idx, cam_idx = linear_sum_assignment(dist)
    return {gage_numbers[g]: cam_locs.loc[c, "site"] for g, c in zip(gage_idx, cam_idx)}

def load_q():
    gage_numbers = ["09470500", "09470920", "09471000", "09471550"]
    gage_camera = pair_gages_to_cameras(gage_numbers)

    streamflow, _ = nwis.get_dv(
        sites = gage_numbers,
        start = "2006-01-01",
        end = "2025-12-31",
        parameterCd = "00060",
        statCd = "00003",
    )
    streamflow = streamflow.reset_index()
    streamflow["date"] = pd.to_datetime(streamflow["datetime"]).dt.tz_localize(None)
    streamflow["site"] = streamflow["site_no"].map(gage_camera)
    streamflow = streamflow.dropna(subset = ["site"])
    return streamflow.rename(columns = {"00060_Mean": "q"})[["site", "date", "q"]]

def load_gw():
    path = os.path.join(dir, "GW", "CamSPRNCA_GW.csv")
    gw = pd.read_csv(path)
    gw.columns = gw.columns.str.strip().str.lower()
    gw["date"] = pd.to_datetime(gw["datetime"], format = "mixed")
    gw = gw.rename(columns = {"dtgw_m": "dtgw"})
    return gw.groupby(["site", "camera", "date"])["dtgw"].mean().reset_index()

def merge_all():
    climate = load_climate()
    et = load_et()
    fp = load_fp()
    gw = load_gw()
    q = load_q()

    hydro = climate.merge(fp[["site", "date", target]], on = ["site", "date"], how = "right")
    hydro = hydro.merge(et[["site", "date", "et"]], on = ["site", "date"], how = "left")
    hydro = hydro.merge(gw[["site", "date", "dtgw"]], on = ["site", "date"], how = "left")
    hydro = hydro.merge(q[["site", "date", "q"]], on = ["site", "date"], how = "left")
    hydro = hydro.sort_values(["site", "date"])
    hydro["dtgw"] = (
        hydro.groupby("site")["dtgw"]
        .transform(lambda s: s.interpolate(method = "linear", limit_direction = "both"))
    )
    hydro["q"] = (
        hydro.groupby("site")["q"]
        .transform(lambda s: s.interpolate(method = "linear", limit_direction = "both"))
    )
    hydro["year"] = hydro["date"].dt.year
    hydro["camera"] = hydro["site"].map(cameras)
    hydro = hydro[hydro["site"].isin(camera_sites)].copy()
    return hydro


In [17]:
class HydroDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.FloatTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def build_feature_sequences(df, features, seq_len, sites):
    sequences, labels, meta = [], [], []
    for site in sites:
        sdata = df[df["site"] == site].sort_values("date").reset_index(drop = True)
        x_vals = sdata[features].values
        y_vals = sdata[target].values
        dates = sdata["date"].values

        for i in range(seq_len, len(sdata)):
            seq = x_vals[i - seq_len:i]
            label = y_vals[i]
            if np.isnan(seq).any() or np.isnan(label):
                continue
            sequences.append(seq)
            labels.append(label)
            meta.append({"site": site, "date": dates[i]})

    return np.array(sequences), np.array(labels), meta

def build_split_sequences(df, features, seq_len, sites, train_flag, val_flag, label):
    train_sequences, train_labels = [], []
    val_sequences, val_labels = [], []
    train_meta, val_meta = [], []

    for site in sites:
        sdata = df[df["site"] == site].sort_values("date").reset_index(drop = True)
        x_vals = sdata[features].values
        y_vals = sdata[label].values
        dates = sdata["date"].values
        is_train = sdata[train_flag].values
        is_val = sdata[val_flag].values

        for i in range(seq_len, len(sdata)):
            seq = x_vals[i - seq_len:i]
            label_val = y_vals[i]
            if np.isnan(seq).any() or np.isnan(label_val):
                continue

            if is_train[i]:
                train_sequences.append(seq)
                train_labels.append(label_val)
                train_meta.append({"site": site, "date": dates[i]})
            elif is_val[i]:
                val_sequences.append(seq)
                val_labels.append(label_val)
                val_meta.append({"site": site, "date": dates[i]})

    if len(train_sequences) == 0:
        train_sequences = np.empty((0, seq_len, len(features)))
        train_labels = np.empty((0,))
    else:
        train_sequences = np.array(train_sequences)
        train_labels = np.array(train_labels)

    if len(val_sequences) == 0:
        val_sequences = np.empty((0, seq_len, len(features)))
        val_labels = np.empty((0,))
    else:
        val_sequences = np.array(val_sequences)
        val_labels = np.array(val_labels)

    return train_sequences, train_labels, train_meta, val_sequences, val_labels, val_meta

def build_prediction_sequences(df, features, seq_len, sites):
    sequences, meta = [], []
    for site in sites:
        sdata = df[df["site"] == site].sort_values("date").reset_index(drop = True)
        x_vals = sdata[features].values
        y_vals = sdata[target].values
        dates = sdata["date"].values

        for i in range(seq_len, len(sdata)):
            if not np.isnan(y_vals[i]):
                continue
            seq = x_vals[i - seq_len:i]
            if np.isnan(seq).any():
                continue
            sequences.append(seq)
            meta.append({"site": site, "date": dates[i]})

    if len(sequences) == 0:
        return np.empty((0, seq_len, len(features))), meta
    return np.array(sequences), meta

In [18]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            bidirectional = True,
            dropout = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out.squeeze(-1)


def train_model(model, train_loader, val_loader, lr, epochs, patience = 10, use_early_stopping = True, trial = None):
    optimizer = torch.optim.Adam(model.parameters(), lr = lr)
    criterion = nn.MSELoss()

    best_rmse = np.inf
    best_state = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

        if val_loader is None or not use_early_stopping:
            continue

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                preds = model(x_batch).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(y_batch.numpy())

        val_rmse = np.sqrt(mean_squared_error(val_true, val_preds))

        if trial is not None:
            trial.report(val_rmse, epoch) # lets the Optuna pruner see incomplete trials and kill bad ones early
            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if use_early_stopping and val_loader is not None and best_state is not None:
        model.load_state_dict(best_state)
    return model, best_rmse


In [ ]:
def suggest_from_space(trial, space):
    params = {}
    for name, spec in space.items():
        if spec["kind"] == "int":
            params[name] = trial.suggest_int(name, spec["low"], spec["high"], step = spec["step"])
        elif spec["kind"] == "float":
            if spec.get("log"):
                params[name] = trial.suggest_float(name, spec["low"], spec["high"], log = True)
            else:
                params[name] = trial.suggest_float(name, spec["low"], spec["high"], step = spec["step"])
        else:
            params[name] = trial.suggest_categorical(name, spec["choices"])
    return params

def expand_if_at_boundary(space, best_params):
    expanded = False
    for name, spec in space.items():
        value = best_params[name]

        if spec["kind"] in ("int", "float"):
            log_scale = spec.get("log", False)
            low, high = spec["low"], spec["high"]
            lo, hi, cur = (np.log10(low), np.log10(high), np.log10(value)) if log_scale else (low, high, value)
            span = hi - lo
            tol = spec["step"] if (spec["kind"] == "int" and not log_scale) else span * 0.02
            shift = span * 0.5

            if cur <= lo + tol:
                new_lo = lo - shift
                new_low = 10 ** new_lo if log_scale else new_lo
                if spec["kind"] == "int" and not log_scale:
                    new_low = round(new_low)
                new_low = max(new_low, spec["floor"])
                if new_low < low:
                    spec["low"] = new_low
                    expanded = True
            elif cur >= hi - tol:
                new_hi = hi + shift
                new_high = 10 ** new_hi if log_scale else new_hi
                if spec["kind"] == "int" and not log_scale:
                    new_high = round(new_high)
                new_high = min(new_high, spec["ceil"])
                if new_high > high:
                    spec["high"] = new_high
                    expanded = True
        else:
            choices = spec["choices"]
            if value == min(choices) and value // 2 >= spec["floor"]:
                spec["choices"] = [value // 2] + choices
                expanded = True
            elif value == max(choices) and value * 2 <= spec["ceil"]:
                spec["choices"] = choices + [value * 2]
                expanded = True

    return expanded

def run_tuning(df, features, n_trials = 60, max_expansions = 2):
    full_df = df.copy()
    train_df = full_df[full_df["is_train"]].copy()

    scaler = StandardScaler()
    scaler.fit(train_df[features])
    scaled_df = full_df.copy()
    scaled_df[features] = scaler.transform(scaled_df[features])

    sequence_cache = {}
    def get_sequences(seq_len):
        if seq_len not in sequence_cache:
            sequence_cache[seq_len] = build_split_sequences(
                scaled_df,
                features,
                seq_len,
                camera_sites,
                train_flag = "is_train",
                val_flag = "is_tune_mask",
                label = "fp_obs",
            )
        return sequence_cache[seq_len]

    space = {
        "seq_len": {"kind": "int", "low": 5, "high": 45, "step": 1, "floor": 2, "ceil": 90},
        "hidden_size": {"kind": "int", "low": 16, "high": 256, "step": 8, "floor": 8, "ceil": 512},
        "num_layers": {"kind": "int", "low": 1, "high": 4, "step": 1, "floor": 1, "ceil": 6},
        "dropout": {"kind": "float", "low": 0.0, "high": 0.6, "step": 0.05, "floor": 0.0, "ceil": 0.9},
        "lr": {"kind": "float", "low": 1e-5, "high": 3e-2, "log": True, "floor": 1e-6, "ceil": 1e-1},
        "batch_size": {"kind": "categorical", "choices": [16, 32, 64, 128, 256], "floor": 8, "ceil": 1024},
    }

    def objective(trial):
        params = suggest_from_space(trial, space)
        train_seqs, train_labels, _, val_seqs, val_labels, _ = get_sequences(params["seq_len"])

        if len(train_seqs) == 0 or len(val_seqs) == 0:
            return np.inf

        train_loader = DataLoader(HydroDataset(train_seqs, train_labels), batch_size = params["batch_size"], shuffle = True)
        val_loader = DataLoader(HydroDataset(val_seqs, val_labels), batch_size = params["batch_size"])

        model = BiLSTMRegressor(
            input_size = len(features),
            hidden_size = params["hidden_size"],
            num_layers = params["num_layers"],
            dropout = params["dropout"],
        ).to(device)

        _, val_rmse = train_model(model, train_loader, val_loader, params["lr"], epochs = 50, patience = 7, trial = trial)
        return val_rmse

    torch.manual_seed(seed)
    np.random.seed(seed)

    study = optuna.create_study(
        direction = "minimize",
        sampler = optuna.samplers.TPESampler(seed = seed),
        pruner = optuna.pruners.MedianPruner(n_startup_trials = 4, n_warmup_steps = 5), # prunes trials that are clearly bad
    )

    trials_this_round = n_trials
    for round_num in range(max_expansions + 1):
        study.optimize(objective, n_trials = trials_this_round, show_progress_bar = True)
        if round_num == max_expansions or not expand_if_at_boundary(space, study.best_params):
            break
        # follow-up rounds only need to explore newly opened edge, not repeat full sweep
        trials_this_round = max(15, n_trials // 3)

    return study.best_params, study.best_value


In [20]:
def evaluate_on_test(model, test_loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            preds = model(x_batch).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(y_batch.numpy())
    return np.array(all_true), np.array(all_preds)

def predict_missing(model, pred_loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for x_batch in pred_loader:
            if isinstance(x_batch, (list, tuple)):
                x_batch = x_batch[0]
            x_batch = x_batch.to(device)
            preds = model(x_batch).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

df = merge_all()
features = predictors.copy()

df = df.dropna(subset = features, how = "all")

df["fp_obs"] = df[target]
observed_pool = df["fp_obs"].notna() & df["site"].isin(camera_sites)
obs_idx = df.index[observed_pool].to_numpy()
n_test = int(len(obs_idx) * test_frac)

rng = np.random.default_rng(seed)
df["is_test_mask"] = False
if n_test > 0:
    test_idx = rng.choice(obs_idx, size = n_test, replace = False)
    df.loc[test_idx, "is_test_mask"] = True

fit_pool = observed_pool & (~df["is_test_mask"])
fit_idx = df.index[fit_pool].to_numpy()
n_tune = int(len(fit_idx) * tune_frac)
df["is_tune_mask"] = False
if n_tune > 0:
    tune_idx = rng.choice(fit_idx, size = n_tune, replace = False)
    df.loc[tune_idx, "is_tune_mask"] = True

df["is_train"] = df["fp_obs"].notna() & (~df["is_test_mask"]) & (~df["is_tune_mask"])
df["is_fit"] = df["fp_obs"].notna() & (~df["is_test_mask"])

train_mask = df["is_train"]
for feat in features:
    n_missing = df[feat].isna().sum()
    if n_missing > 0:
        fill_val = df.loc[train_mask, feat].median() # ensures features are complete
        df[feat] = df[feat].fillna(fill_val)

df.loc[df["is_tune_mask"] | df["is_test_mask"], target] = np.nan

best_params, best_rmse = run_tuning(df, features, n_trials = 60)
best_params, best_rmse, int(df["is_tune_mask"].sum()), int(df["is_test_mask"].sum())

HTTP Request: GET https://waterservices.usgs.gov/nwis/site?sites=09470500%2C09470920%2C09471000%2C09471550&siteOutput=Expanded&format=rdb "HTTP/1.1 200 "
HTTP Request: GET https://waterservices.usgs.gov/nwis/dv?format=json&parameterCd=00060&statCd=00003&startDT=2006-01-01&endDT=2025-12-31&sites=09470500%2C09470920%2C09471000%2C09471550 "HTTP/1.1 200 "


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

({'seq_len': 32,
  'hidden_size': 120,
  'num_layers': 3,
  'dropout': 0.05,
  'lr': 0.0007059301711879509,
  'batch_size': 64},
 0.12804271151778035,
 7326,
 9157)

In [ ]:
params_path = os.path.join(dir, "HydroParams.csv")
params_row = pd.DataFrame([{**best_params, "best_rmse": best_rmse}])
params_row.to_csv(params_path, index = False)


In [ ]:
best_params = pd.read_csv(os.path.join(dir, "HydroParams.csv")).to_dict("records")[0]

seq_len = int(best_params["seq_len"])
hidden_size = int(best_params["hidden_size"])
num_layers = int(best_params["num_layers"])
dropout = float(best_params["dropout"])
lr = float(best_params["lr"])
batch_size = int(best_params["batch_size"])

fit_df = df[df["is_fit"]].copy()
scaler = StandardScaler()
scaler.fit(fit_df[features])

df_scaled = df.copy()
df_scaled[features] = scaler.transform(df_scaled[features])

train_seqs, train_labels, train_meta, test_seqs, test_labels, test_meta = build_split_sequences(
    df_scaled,
    features,
    seq_len,
    camera_sites,
    train_flag = "is_fit",
    val_flag = "is_test_mask",
    label = "fp_obs",
)

train_loader = DataLoader(HydroDataset(train_seqs, train_labels), batch_size = batch_size, shuffle = True)
test_loader = DataLoader(HydroDataset(test_seqs, test_labels), batch_size = batch_size)

torch.manual_seed(seed)
np.random.seed(seed)

model = BiLSTMRegressor(
    input_size = len(features),
    hidden_size = hidden_size,
    num_layers = num_layers,
    dropout = dropout
).to(device)

model, _ = train_model(model, train_loader, None, lr, epochs = 100, patience = 15, use_early_stopping = False)
y_true, y_pred = evaluate_on_test(model, test_loader)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Tuning-mask points: {int(df['is_tune_mask'].sum())}")
print(f"Untouched test-mask points: {int(df['is_test_mask'].sum())}")
print(f"Evaluation sequences: {len(test_seqs)}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R^2:  {r2:.4f}")

In [ ]:
df_final = df_scaled.copy()
df_final[target] = df_final["fp_obs"]

all_seqs, all_labels, _ = build_feature_sequences(df_final, features, seq_len, camera_sites)
all_loader = DataLoader(HydroDataset(all_seqs, all_labels), batch_size = batch_size, shuffle = True)

final_model = BiLSTMRegressor(
    input_size = len(features),
    hidden_size = hidden_size,
    num_layers = num_layers,
    dropout = dropout
).to(device)

final_model, _ = train_model(final_model, all_loader, all_loader, lr, epochs = 100, patience = 15)

pred_target_df = df_final.copy()
pred_target_df[target] = pred_target_df["fp_obs"]
pred_target_df.loc[pred_target_df["fp_obs"].notna(), target] = pred_target_df["fp_obs"]

pred_seqs, pred_meta = build_prediction_sequences(pred_target_df, features, seq_len, camera_sites)

if len(pred_seqs) > 0:
    pred_dataset = torch.FloatTensor(pred_seqs)
    pred_loader = DataLoader(pred_dataset, batch_size = batch_size)
    preds = predict_missing(final_model, pred_loader)
    preds = np.clip(np.rint(preds), 0, 1).astype(int)

    pred_df = pd.DataFrame(pred_meta)
    pred_df[f"{target}_pred"] = preds
    pred_df["date"] = pd.to_datetime(pred_df["date"])
else:
    pred_df = pd.DataFrame(columns = ["site", "date", f"{target}_pred"])

out = df_final[["site", "camera", "date", "prcp", "vpd", "tmax", "tmin", "fp_obs", "dtgw"]].copy()
out = out.rename(columns = {"fp_obs": target})
out = out.merge(pred_df[["site", "date", f"{target}_pred"]], on = ["site", "date"], how = "left")
out[f"{target}_final"] = out[target].fillna(out[f"{target}_pred"]).astype("Int64")
out["source"] = out[target].apply(lambda x: "observed" if pd.notna(x) else "predicted")
out.loc[out[target].isna() & out[f"{target}_pred"].notna(), "source"] = "predicted"

out_path = os.path.join(dir, f"Pred{target.upper()}.csv")
out.to_csv(out_path, index = False)
out_path, len(out), (out["source"] == "predicted").sum(), out[f"{target}_final"].isna().sum()

In [ ]:
test_pred_df = pd.DataFrame(test_meta)
test_pred_df["date"] = pd.to_datetime(test_pred_df["date"])
test_pred_df["fp_test_pred"] = y_pred

plot_df = out.merge(test_pred_df[["site", "date", "fp_test_pred"]], on = ["site", "date"], how = "left")
plot_df["date"] = pd.to_datetime(plot_df["date"])

sites_sorted = sorted(camera_sites)
fig, axes = plt.subplots(len(sites_sorted), 1, figsize = (18, 2.2 * len(sites_sorted)), sharex = True)
if len(sites_sorted) == 1:
    axes = [axes]

for idx, site in enumerate(sites_sorted):
    ax = axes[idx]
    sdata = plot_df[plot_df["site"] == site].sort_values("date")

    obs = sdata[sdata["source"] == "observed"]
    pred = sdata[sdata["source"] == "predicted"]

    obs_flow = obs[obs[target] == 1]
    obs_noflow = obs[obs[target] == 0]
    ax.vlines(obs_flow["date"], 0, 1, colors = "#2166ac", linewidth = 0.7, alpha = 0.9)
    ax.vlines(obs_noflow["date"], 0, -1, colors = "#e6b800", linewidth = 0.5, alpha = 0.9)

    pred_flow = pred[pred[f"{target}_pred"] == 1]
    pred_noflow = pred[pred[f"{target}_pred"] == 0]
    ax.vlines(pred_flow["date"], 0, 1, colors = "#92c5de", linewidth = 0.55, alpha = 0.9)
    ax.vlines(pred_noflow["date"], 0, -1, colors = "#fff3a8", linewidth = 0.4, alpha = 0.9)

    test = sdata[sdata["fp_test_pred"].notna()].copy()
    test["pred_bin"] = (test["fp_test_pred"] > 0.5).astype(int)
    test["correct"] = test["pred_bin"] == test[target].astype(int)

    ax.scatter(
        test[test["correct"]]["date"],
        [0] * test["correct"].sum(),
        color = "#1a9850",
        s = 22,
        zorder = 5,
        marker = "o"
    )
    ax.scatter(
        test[~test["correct"]]["date"],
        [0] * (~test["correct"]).sum(),
        color = "#d7191c",
        s = 34,
        zorder = 6,
        marker = "x",
        linewidths = 1.2
    )

    ax.axhline(0, color = "black", linewidth = 0.5, linestyle = "--")
    ax.set_yticklabels(["0", "", "1"])
    ax.set_yticks([-1, 0, 1])
    ax.set_ylabel("FP", fontsize = 16, labelpad = 6)
    formatted_site = re.sub(r"(?<=[a-z\.])(?=[A-Z])", " ", site)
    ax.set_title(formatted_site, fontsize = 20, fontweight = "normal", loc = "left")
    ax.set_xlim(sdata["date"].min(), sdata["date"].max())
    ax.tick_params(axis = "x", labelsize = 16)
    ax.spines[["top", "right"]].set_visible(False)

legend_handles = [
    mpatches.Patch(color = "#2166ac", label = "Wet"),
    mpatches.Patch(color = "#e6b800", label = "Dry"),
    mpatches.Patch(color = "#92c5de", label = "Predicted Wet"),
    mpatches.Patch(color = "#fff3a8", label = "Predicted Dry"),
    plt.scatter([], [], color = "#1a9850", s = 32, marker = "o", label = "Correct Test"),
    plt.scatter([], [], color = "#d7191c", s = 45, marker = "x", linewidths = 1.4, label = "Incorrect Test"),
]
fig.legend(
    handles = legend_handles,
    loc = "upper center",
    ncol = 6,
    fontsize = 20,
    bbox_to_anchor = (0.5, 0.96),
    frameon = False,
    borderpad = 0.08,
    borderaxespad = 0.02,
    handletextpad = 0.25,
    columnspacing = 0.45,
    handlelength = 1.1
)

plt.suptitle("Flow Persistence BiLSTM Predictions", fontsize = 24, y = 0.992)
plt.tight_layout(rect = [0, 0.03, 1, 0.98])

%config InlineBackend.figure_format = "retina"

# plt.savefig("../Figures/bilstm_ts.png", format = "png", dpi = 300, bbox_inches = "tight")